# 06 — Implied Volatility Inversion and No-Arbitrage Bounds

This notebook studies implied volatility as the inverse problem of Black-Scholes-Merton pricing.

The previous notebooks established the no-arbitrage foundation, Brownian motion and GBM, Itô's lemma, the Black-Scholes PDE, the closed-form Black-Scholes-Merton formula, Greeks, risk-neutral Monte Carlo pricing, and discrete delta-hedging error.

This notebook now asks a different question:

> Given an observed option price, when does a valid Black-Scholes implied volatility exist, and how can it be recovered reliably?

The key point is that implied volatility is not automatically meaningful. Before attempting numerical inversion, the observed option price must satisfy European no-arbitrage bounds. If the price is below its lower bound, above its upper bound, too close to intrinsic value, or located in a very low-vega region, the implied volatility may be invalid, degenerate, or numerically unstable.

This notebook remains synthetic and theory-first. It does not use real option-chain data yet.

## Scope

In scope:

- European call and put no-arbitrage bounds
- Put-call parity with continuous dividend yield
- Discounted intrinsic value and time value
- Black-Scholes-Merton price ranges as volatility varies
- Vega and uniqueness of implied volatility
- Implied volatility inversion as a root-finding problem
- Synthetic flat-vol recovery tests
- Invalid price diagnostics
- Near-expiry and low-vega failure modes
- Synthetic smile construction and recovery
- Final validation ledger

Out of scope:

- real option chains
- bid/ask spreads
- early exercise
- static arbitrage across full option grids
- SVI or SSVI fitting
- Heston or Bates calibration
- market-data cleaning

## Core claim

A single option price can be inverted into Black-Scholes implied volatility only when it is economically valid and numerically well-conditioned.

Notebook 06 answers:

> Is each individual option price valid and invertible?

The next notebook will answer:

> Once individual implied volatilities are valid, does the full option grid violate static arbitrage?

In [1]:
# ============================================================
# Notebook 06 setup
# Implied volatility inversion and no-arbitrage bounds
# ============================================================

from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from math import exp, isfinite, log, sqrt
from typing import Any, Literal

import numpy as np
import pandas as pd

from scipy.optimize import brentq
from scipy.stats import norm


# ------------------------------------------------------------
# Display and numerical settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.8f}")

RNG_SEED: int = 620_006
rng = np.random.default_rng(RNG_SEED)

FLOAT_TOL: float = 1e-12
PRICE_TOL: float = 1e-10
IV_TOL: float = 1e-8
LOW_VEGA_TOL: float = 1e-8

SIGMA_MIN: float = 1e-8
SIGMA_MAX_DEFAULT: float = 5.0
SIGMA_MAX_HARD: float = 10.0


# ------------------------------------------------------------
# Option type convention
# ------------------------------------------------------------

class OptionType(str, Enum):
    CALL = "call"
    PUT = "put"


# ------------------------------------------------------------
# Baseline Black-Scholes-Merton configuration
# ------------------------------------------------------------

@dataclass(frozen=True)
class BSMParams:
    """
    Parameters for a European option under the Black-Scholes-Merton setup.

    S     : spot price
    K     : strike price
    tau   : time to maturity in years
    r     : continuously compounded risk-free rate
    q     : continuously compounded dividend yield
    sigma : volatility
    """

    S: float = 100.0
    K: float = 100.0
    tau: float = 1.0
    r: float = 0.05
    q: float = 0.00
    sigma: float = 0.20

    def validate(self) -> None:
        if not isfinite(self.S) or self.S <= 0:
            raise ValueError("S must be positive and finite.")
        if not isfinite(self.K) or self.K <= 0:
            raise ValueError("K must be positive and finite.")
        if not isfinite(self.tau) or self.tau < 0:
            raise ValueError("tau must be nonnegative and finite.")
        if not isfinite(self.r):
            raise ValueError("r must be finite.")
        if not isfinite(self.q):
            raise ValueError("q must be finite.")
        if not isfinite(self.sigma) or self.sigma < 0:
            raise ValueError("sigma must be nonnegative and finite.")


BASE = BSMParams()
BASE.validate()


# ------------------------------------------------------------
# Validation ledger helper
# ------------------------------------------------------------

validation_rows: list[dict[str, Any]] = []


def record_check(
    check: str,
    passed: bool,
    detail: str,
    value: Any | None = None,
) -> None:
    """
    Append one row to the notebook validation ledger.
    """
    validation_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "value": value,
        }
    )


def show_validation_ledger() -> pd.DataFrame:
    """
    Return the current validation ledger as a DataFrame.
    """
    if not validation_rows:
        return pd.DataFrame(columns=["check", "passed", "detail", "value"])

    return pd.DataFrame(validation_rows)


print("Notebook 06 setup complete.")
print(f"Baseline parameters: {BASE}")

Notebook 06 setup complete.
Baseline parameters: BSMParams(S=100.0, K=100.0, tau=1.0, r=0.05, q=0.0, sigma=0.2)


## 1. European no-arbitrage bounds

Before implied volatility can be computed, the option price must first pass basic European no-arbitrage bounds.

Let:

$$
D_r = e^{-r\tau}
$$

$$
D_q = e^{-q\tau}
$$

where:

- $D_r$ is the risk-free discount factor,
- $D_q$ is the dividend discount factor,
- $S D_q$ is the present value of the stock exposure,
- $K D_r$ is the present value of the strike payment.

For a European call option, the no-arbitrage bounds are:

$$
\max(SD_q - KD_r, 0) \leq C \leq SD_q
$$

For a European put option, the no-arbitrage bounds are:

$$
\max(KD_r - SD_q, 0) \leq P \leq KD_r
$$

Put-call parity is:

$$
C - P = SD_q - KD_r
$$

These conditions are not optional. If an observed option price violates them, the problem is not that the implied volatility is high or low. The problem is that the price is invalid under the European Black-Scholes-Merton assumptions.

This notebook therefore treats no-arbitrage bounds as the first gate before numerical inversion.

In [2]:
# ============================================================
# European no-arbitrage bounds and put-call parity
# ============================================================

def discount_factors(tau: float, r: float, q: float) -> tuple[float, float]:
    """
    Return the risk-free and dividend discount factors.

    Dr = exp(-r tau)
    Dq = exp(-q tau)
    """
    if tau < 0:
        raise ValueError("tau must be nonnegative.")

    Dr = exp(-r * tau)
    Dq = exp(-q * tau)
    return Dr, Dq


def option_bounds(
    option_type: OptionType | str,
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
) -> tuple[float, float]:
    """
    Return European no-arbitrage lower and upper bounds.

    Call:
        max(S Dq - K Dr, 0) <= C <= S Dq

    Put:
        max(K Dr - S Dq, 0) <= P <= K Dr
    """
    option_type = OptionType(option_type)

    if S <= 0 or K <= 0:
        raise ValueError("S and K must be positive.")
    if tau < 0:
        raise ValueError("tau must be nonnegative.")

    Dr, Dq = discount_factors(tau=tau, r=r, q=q)

    stock_pv = S * Dq
    strike_pv = K * Dr

    if option_type == OptionType.CALL:
        lower = max(stock_pv - strike_pv, 0.0)
        upper = stock_pv
    elif option_type == OptionType.PUT:
        lower = max(strike_pv - stock_pv, 0.0)
        upper = strike_pv
    else:
        raise ValueError(f"Unsupported option type: {option_type}")

    return lower, upper


def discounted_intrinsic_value(
    option_type: OptionType | str,
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
) -> float:
    """
    Return the discounted intrinsic value.

    For European options under continuous dividend yield, this is the
    no-arbitrage lower bound.
    """
    lower, _ = option_bounds(
        option_type=option_type,
        S=S,
        K=K,
        tau=tau,
        r=r,
        q=q,
    )
    return lower


def time_value(
    option_type: OptionType | str,
    price: float,
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
) -> float:
    """
    Return option time value relative to the discounted intrinsic value.
    """
    intrinsic = discounted_intrinsic_value(
        option_type=option_type,
        S=S,
        K=K,
        tau=tau,
        r=r,
        q=q,
    )
    return price - intrinsic


def put_call_parity_gap(
    call_price: float,
    put_price: float,
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
) -> float:
    """
    Return the put-call parity gap.

    A valid European call-put pair should satisfy:

        C - P = S Dq - K Dr

    Therefore, this function returns:

        gap = (C - P) - (S Dq - K Dr)

    A gap near zero means parity holds.
    """
    Dr, Dq = discount_factors(tau=tau, r=r, q=q)

    parity_value = S * Dq - K * Dr
    observed_value = call_price - put_price

    return observed_value - parity_value


def bounds_diagnostic_row(
    option_type: OptionType | str,
    price: float,
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
    price_tol: float = PRICE_TOL,
) -> dict[str, Any]:
    """
    Return a diagnostic row for a single option price.
    """
    option_type = OptionType(option_type)

    lower, upper = option_bounds(
        option_type=option_type,
        S=S,
        K=K,
        tau=tau,
        r=r,
        q=q,
    )

    intrinsic = lower
    tv = price - intrinsic

    below_lower = price < lower - price_tol
    above_upper = price > upper + price_tol
    near_lower = abs(price - lower) <= price_tol
    near_upper = abs(price - upper) <= price_tol

    if below_lower:
        status = "BELOW_LOWER_BOUND"
    elif above_upper:
        status = "ABOVE_UPPER_BOUND"
    elif near_lower:
        status = "AT_INTRINSIC_OR_ZERO_VOL"
    elif near_upper:
        status = "NEAR_UPPER_BOUND"
    else:
        status = "VALID"

    return {
        "option_type": option_type.value,
        "price": price,
        "S": S,
        "K": K,
        "tau": tau,
        "r": r,
        "q": q,
        "lower_bound": lower,
        "upper_bound": upper,
        "discounted_intrinsic": intrinsic,
        "time_value": tv,
        "status": status,
        "passes_bounds": status == "VALID",
    }


# ------------------------------------------------------------
# Baseline bounds demonstration
# ------------------------------------------------------------

call_lower, call_upper = option_bounds(
    option_type=OptionType.CALL,
    S=BASE.S,
    K=BASE.K,
    tau=BASE.tau,
    r=BASE.r,
    q=BASE.q,
)

put_lower, put_upper = option_bounds(
    option_type=OptionType.PUT,
    S=BASE.S,
    K=BASE.K,
    tau=BASE.tau,
    r=BASE.r,
    q=BASE.q,
)

bounds_demo = pd.DataFrame(
    [
        {
            "option_type": "call",
            "lower_bound": call_lower,
            "upper_bound": call_upper,
        },
        {
            "option_type": "put",
            "lower_bound": put_lower,
            "upper_bound": put_upper,
        },
    ]
)

record_check(
    check="call_bounds_order",
    passed=0.0 <= call_lower <= call_upper,
    detail="European call lower bound should be nonnegative and no greater than upper bound.",
    value={"lower": call_lower, "upper": call_upper},
)

record_check(
    check="put_bounds_order",
    passed=0.0 <= put_lower <= put_upper,
    detail="European put lower bound should be nonnegative and no greater than upper bound.",
    value={"lower": put_lower, "upper": put_upper},
)

bounds_demo

,option_type,lower_bound,upper_bound
0,call,4.87705755,100.00000000
1,put,0.00000000,95.12294245


## 2. Black-Scholes-Merton price range as volatility varies

The no-arbitrage bounds are also the limiting price range of the Black-Scholes-Merton formula.

For a fixed option contract, let volatility vary while keeping \(S\), \(K\), \(\tau\), \(r\), and \(q\) fixed.

As volatility approaches zero, the option price approaches its discounted intrinsic value:

$$
\lim_{\sigma \to 0} C_{\mathrm{BSM}}(\sigma)
=
\max(SD_q - KD_r, 0)
$$

$$
\lim_{\sigma \to 0} P_{\mathrm{BSM}}(\sigma)
=
\max(KD_r - SD_q, 0)
$$

As volatility becomes extremely large, the option price approaches its upper bound:

$$
\lim_{\sigma \to \infty} C_{\mathrm{BSM}}(\sigma)
=
SD_q
$$

$$
\lim_{\sigma \to \infty} P_{\mathrm{BSM}}(\sigma)
=
KD_r
$$

Therefore, an observed option price can only have a Black-Scholes implied volatility if it lies inside the corresponding price range.

For calls:

$$
\max(SD_q - KD_r, 0)
\leq
C_{\mathrm{obs}}
\leq
SD_q
$$

For puts:

$$
\max(KD_r - SD_q, 0)
\leq
P_{\mathrm{obs}}
\leq
KD_r
$$

Interior prices can be inverted into positive implied volatility. Boundary prices are degenerate: they correspond to zero volatility, extremely large volatility, or numerical instability.

In [3]:
# ============================================================
# Black-Scholes-Merton pricing and volatility-limit checks
# ============================================================

def _as_array(x: float | np.ndarray) -> np.ndarray:
    """
    Convert scalar or array-like input to a NumPy array.
    """
    return np.asarray(x, dtype=float)


def bsm_d1_d2(
    S: float | np.ndarray,
    K: float | np.ndarray,
    tau: float | np.ndarray,
    r: float,
    q: float,
    sigma: float | np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Return d1 and d2 for the Black-Scholes-Merton formula.

    This function assumes strictly positive tau and sigma.
    Boundary cases are handled in the pricing wrapper.
    """
    S_arr = _as_array(S)
    K_arr = _as_array(K)
    tau_arr = _as_array(tau)
    sigma_arr = _as_array(sigma)

    sqrt_tau = np.sqrt(tau_arr)

    d1 = (
        np.log(S_arr / K_arr)
        + (r - q + 0.5 * sigma_arr**2) * tau_arr
    ) / (sigma_arr * sqrt_tau)

    d2 = d1 - sigma_arr * sqrt_tau

    return d1, d2


def bsm_price(
    option_type: OptionType | str,
    S: float | np.ndarray,
    K: float | np.ndarray,
    tau: float | np.ndarray,
    r: float,
    q: float,
    sigma: float | np.ndarray,
) -> np.ndarray:
    """
    Black-Scholes-Merton price for European calls and puts.

    Handles the degenerate cases tau = 0 or sigma = 0 by returning
    discounted intrinsic value.
    """
    option_type = OptionType(option_type)

    S_arr = _as_array(S)
    K_arr = _as_array(K)
    tau_arr = _as_array(tau)
    sigma_arr = _as_array(sigma)

    S_b, K_b, tau_b, sigma_b = np.broadcast_arrays(
        S_arr,
        K_arr,
        tau_arr,
        sigma_arr,
    )

    price = np.empty_like(S_b, dtype=float)

    degenerate = (tau_b <= FLOAT_TOL) | (sigma_b <= FLOAT_TOL)
    regular = ~degenerate

    if np.any(degenerate):
        Dr_deg = np.exp(-r * tau_b[degenerate])
        Dq_deg = np.exp(-q * tau_b[degenerate])

        if option_type == OptionType.CALL:
            price[degenerate] = np.maximum(
                S_b[degenerate] * Dq_deg - K_b[degenerate] * Dr_deg,
                0.0,
            )
        else:
            price[degenerate] = np.maximum(
                K_b[degenerate] * Dr_deg - S_b[degenerate] * Dq_deg,
                0.0,
            )

    if np.any(regular):
        d1, d2 = bsm_d1_d2(
            S=S_b[regular],
            K=K_b[regular],
            tau=tau_b[regular],
            r=r,
            q=q,
            sigma=sigma_b[regular],
        )

        Dr = np.exp(-r * tau_b[regular])
        Dq = np.exp(-q * tau_b[regular])

        if option_type == OptionType.CALL:
            price[regular] = (
                S_b[regular] * Dq * norm.cdf(d1)
                - K_b[regular] * Dr * norm.cdf(d2)
            )
        else:
            price[regular] = (
                K_b[regular] * Dr * norm.cdf(-d2)
                - S_b[regular] * Dq * norm.cdf(-d1)
            )

    return price


def bsm_vega(
    S: float | np.ndarray,
    K: float | np.ndarray,
    tau: float | np.ndarray,
    r: float,
    q: float,
    sigma: float | np.ndarray,
) -> np.ndarray:
    """
    Black-Scholes-Merton vega.

    Vega is the derivative of option price with respect to volatility.
    It is the same for calls and puts under BSM.
    """
    S_arr = _as_array(S)
    K_arr = _as_array(K)
    tau_arr = _as_array(tau)
    sigma_arr = _as_array(sigma)

    S_b, K_b, tau_b, sigma_b = np.broadcast_arrays(
        S_arr,
        K_arr,
        tau_arr,
        sigma_arr,
    )

    vega = np.zeros_like(S_b, dtype=float)

    regular = (tau_b > FLOAT_TOL) & (sigma_b > FLOAT_TOL)

    if np.any(regular):
        d1, _ = bsm_d1_d2(
            S=S_b[regular],
            K=K_b[regular],
            tau=tau_b[regular],
            r=r,
            q=q,
            sigma=sigma_b[regular],
        )

        Dq = np.exp(-q * tau_b[regular])

        vega[regular] = (
            S_b[regular]
            * Dq
            * norm.pdf(d1)
            * np.sqrt(tau_b[regular])
        )

    return vega


# ------------------------------------------------------------
# Baseline BSM prices and parity check
# ------------------------------------------------------------

baseline_call = float(
    bsm_price(
        option_type=OptionType.CALL,
        S=BASE.S,
        K=BASE.K,
        tau=BASE.tau,
        r=BASE.r,
        q=BASE.q,
        sigma=BASE.sigma,
    )
)

baseline_put = float(
    bsm_price(
        option_type=OptionType.PUT,
        S=BASE.S,
        K=BASE.K,
        tau=BASE.tau,
        r=BASE.r,
        q=BASE.q,
        sigma=BASE.sigma,
    )
)

baseline_parity_gap = put_call_parity_gap(
    call_price=baseline_call,
    put_price=baseline_put,
    S=BASE.S,
    K=BASE.K,
    tau=BASE.tau,
    r=BASE.r,
    q=BASE.q,
)

record_check(
    check="baseline_put_call_parity",
    passed=abs(baseline_parity_gap) <= 1e-10,
    detail="Synthetic BSM call and put prices should satisfy put-call parity.",
    value=baseline_parity_gap,
)


# ------------------------------------------------------------
# Volatility range demonstration
# ------------------------------------------------------------

sigma_grid = np.array(
    [
        0.0,
        1e-8,
        0.01,
        0.05,
        0.10,
        0.20,
        0.40,
        0.80,
        1.50,
        3.00,
        5.00,
        10.00,
    ],
    dtype=float,
)

call_prices_by_sigma = bsm_price(
    option_type=OptionType.CALL,
    S=BASE.S,
    K=BASE.K,
    tau=BASE.tau,
    r=BASE.r,
    q=BASE.q,
    sigma=sigma_grid,
)

put_prices_by_sigma = bsm_price(
    option_type=OptionType.PUT,
    S=BASE.S,
    K=BASE.K,
    tau=BASE.tau,
    r=BASE.r,
    q=BASE.q,
    sigma=sigma_grid,
)

vega_by_sigma = bsm_vega(
    S=BASE.S,
    K=BASE.K,
    tau=BASE.tau,
    r=BASE.r,
    q=BASE.q,
    sigma=sigma_grid,
)

price_range_demo = pd.DataFrame(
    {
        "sigma": sigma_grid,
        "call_price": call_prices_by_sigma,
        "put_price": put_prices_by_sigma,
        "vega": vega_by_sigma,
        "call_lower_bound": call_lower,
        "call_upper_bound": call_upper,
        "put_lower_bound": put_lower,
        "put_upper_bound": put_upper,
    }
)

call_monotone = bool(np.all(np.diff(call_prices_by_sigma) >= -PRICE_TOL))
put_monotone = bool(np.all(np.diff(put_prices_by_sigma) >= -PRICE_TOL))

call_inside_bounds = bool(
    np.all(call_prices_by_sigma >= call_lower - PRICE_TOL)
    and np.all(call_prices_by_sigma <= call_upper + PRICE_TOL)
)

put_inside_bounds = bool(
    np.all(put_prices_by_sigma >= put_lower - PRICE_TOL)
    and np.all(put_prices_by_sigma <= put_upper + PRICE_TOL)
)

record_check(
    check="call_price_monotone_in_sigma",
    passed=call_monotone,
    detail="European call BSM price should be nondecreasing in volatility.",
    value=None,
)

record_check(
    check="put_price_monotone_in_sigma",
    passed=put_monotone,
    detail="European put BSM price should be nondecreasing in volatility.",
    value=None,
)

record_check(
    check="call_prices_inside_bounds",
    passed=call_inside_bounds,
    detail="BSM call prices across the volatility grid should remain inside no-arbitrage bounds.",
    value=None,
)

record_check(
    check="put_prices_inside_bounds",
    passed=put_inside_bounds,
    detail="BSM put prices across the volatility grid should remain inside no-arbitrage bounds.",
    value=None,
)

price_range_demo

,sigma,call_price,put_price,vega,call_lower_bound,call_upper_bound,put_lower_bound,put_upper_bound
0,0.00000000,4.87705755,0.00000000,0.00000000,4.87705755,100.00000000,0.00000000,95.12294245
1,0.00000001,4.87705755,0.00000000,0.00000000,4.87705755,100.00000000,0.00000000,95.12294245
2,0.01000000,4.87705760,0.00000005,0.00014500,4.87705755,100.00000000,0.00000000,95.12294245
3,0.05000000,5.28326899,0.40621144,23.59227087,4.87705755,100.00000000,0.00000000,95.12294245
4,0.10000000,6.80495771,1.92790016,34.29438550,4.87705755,100.00000000,0.00000000,95.12294245
5,0.20000000,10.45058357,5.57352602,37.52403469,4.87705755,100.00000000,0.00000000,95.12294245
6,0.40000000,18.02295145,13.14589390,37.84198319,4.87705755,100.00000000,0.00000000,95.12294245
7,0.80000000,32.82098247,27.94392492,35.84766842,4.87705755,100.00000000,0.00000000,95.12294245
8,1.50000000,55.80427837,50.92722082,29.35391998,4.87705755,100.00000000,0.00000000,95.12294245
9,3.00000000,86.96964579,82.09258824,12.63022516,4.87705755,100.00000000,0.00000000,95.12294245


## 3. Vega and uniqueness of implied volatility

The previous table shows that Black-Scholes-Merton prices increase as volatility increases.

This matters because implied volatility is an inverse problem. Given an observed price, we want to find the volatility input that reproduces it.

For a call:

$$
C_{\mathrm{BSM}}(S,K,\tau,r,q,\sigma_{\mathrm{imp}})
=
C_{\mathrm{obs}}
$$

For a put:

$$
P_{\mathrm{BSM}}(S,K,\tau,r,q,\sigma_{\mathrm{imp}})
=
P_{\mathrm{obs}}
$$

The reason this inversion is usually well-defined is vega.

Vega is the sensitivity of the option price to volatility:

$$
\mathrm{Vega}
=
\frac{\partial V}{\partial \sigma}
$$

Under Black-Scholes-Merton, call and put vega are the same:

$$
\mathrm{Vega}
=
S e^{-q\tau} \phi(d_1)\sqrt{\tau}
$$

where $\phi(d_1)$ is the standard normal density evaluated at $d_1$.

For ordinary cases with positive spot, positive strike, positive time to maturity, and positive volatility, vega is nonnegative. In the main interior region, vega is positive, so the Black-Scholes-Merton price is increasing in volatility.

That gives the key implication:

> If an observed option price lies strictly inside the valid Black-Scholes-Merton price range, then its implied volatility is unique.

However, uniqueness does not guarantee numerical stability. Vega can become extremely small for:

- very short maturities,
- deep out-of-the-money options,
- deep in-the-money options,
- prices very close to intrinsic value,
- prices very close to the upper bound.

In those regions, a tiny price change can create a large implied-volatility change.

The approximate local relationship is:

$$
\Delta \sigma
\approx
\frac{\Delta V}{\mathrm{Vega}}
$$

So when vega is small, implied volatility becomes fragile even if the price technically passes no-arbitrage bounds.

In [4]:
# ============================================================
# Implied volatility inversion
# ============================================================

def implied_volatility(
    option_type: OptionType | str,
    price: float,
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
    sigma_min: float = SIGMA_MIN,
    sigma_max: float = SIGMA_MAX_DEFAULT,
    sigma_max_hard: float = SIGMA_MAX_HARD,
    price_tol: float = PRICE_TOL,
    low_vega_tol: float = LOW_VEGA_TOL,
) -> dict[str, Any]:
    """
    Recover Black-Scholes-Merton implied volatility using a bracketed root solver.

    The function first checks European no-arbitrage bounds. If the price is outside
    the valid range, no implied volatility is returned.

    Boundary prices are labeled as degenerate instead of forced through inversion.
    """
    option_type = OptionType(option_type)

    lower, upper = option_bounds(
        option_type=option_type,
        S=S,
        K=K,
        tau=tau,
        r=r,
        q=q,
    )

    diagnostic = bounds_diagnostic_row(
        option_type=option_type,
        price=price,
        S=S,
        K=K,
        tau=tau,
        r=r,
        q=q,
        price_tol=price_tol,
    )

    if not isfinite(price):
        return {
            **diagnostic,
            "implied_vol": np.nan,
            "vega_at_iv": np.nan,
            "iv_status": "INVALID_PRICE_NONFINITE",
            "iv_error": "Observed price is not finite.",
        }

    if price < lower - price_tol:
        return {
            **diagnostic,
            "implied_vol": np.nan,
            "vega_at_iv": np.nan,
            "iv_status": "NO_IV_BELOW_LOWER_BOUND",
            "iv_error": "Observed price is below the European no-arbitrage lower bound.",
        }

    if price > upper + price_tol:
        return {
            **diagnostic,
            "implied_vol": np.nan,
            "vega_at_iv": np.nan,
            "iv_status": "NO_IV_ABOVE_UPPER_BOUND",
            "iv_error": "Observed price is above the European no-arbitrage upper bound.",
        }

    if tau <= FLOAT_TOL:
        return {
            **diagnostic,
            "implied_vol": np.nan,
            "vega_at_iv": np.nan,
            "iv_status": "NO_IV_EXPIRED_OPTION",
            "iv_error": "Option is expired or effectively expired.",
        }

    if abs(price - lower) <= price_tol:
        return {
            **diagnostic,
            "implied_vol": 0.0,
            "vega_at_iv": 0.0,
            "iv_status": "ZERO_VOL_DEGENERATE",
            "iv_error": None,
        }

    if abs(price - upper) <= price_tol:
        return {
            **diagnostic,
            "implied_vol": np.inf,
            "vega_at_iv": 0.0,
            "iv_status": "INFINITE_VOL_DEGENERATE",
            "iv_error": None,
        }

    def objective(sigma: float) -> float:
        model_price = float(
            bsm_price(
                option_type=option_type,
                S=S,
                K=K,
                tau=tau,
                r=r,
                q=q,
                sigma=sigma,
            )
        )
        return model_price - price

    f_low = objective(sigma_min)
    f_high = objective(sigma_max)

    expanded_sigma_max = sigma_max

    while f_low * f_high > 0 and expanded_sigma_max < sigma_max_hard:
        expanded_sigma_max *= 2.0
        expanded_sigma_max = min(expanded_sigma_max, sigma_max_hard)
        f_high = objective(expanded_sigma_max)

    if f_low * f_high > 0:
        return {
            **diagnostic,
            "implied_vol": np.nan,
            "vega_at_iv": np.nan,
            "iv_status": "ROOT_NOT_BRACKETED",
            "iv_error": (
                "Could not bracket the implied-volatility root inside the allowed volatility range."
            ),
        }

    try:
        iv = brentq(
            objective,
            sigma_min,
            expanded_sigma_max,
            xtol=IV_TOL,
            rtol=IV_TOL,
            maxiter=200,
        )

        vega_at_iv = float(
            bsm_vega(
                S=S,
                K=K,
                tau=tau,
                r=r,
                q=q,
                sigma=iv,
            )
        )

        if vega_at_iv < low_vega_tol:
            iv_status = "VALID_LOW_VEGA_WARNING"
            iv_error = "Implied volatility recovered, but vega is very small."
        else:
            iv_status = "VALID_IV"
            iv_error = None

        return {
            **diagnostic,
            "implied_vol": iv,
            "vega_at_iv": vega_at_iv,
            "iv_status": iv_status,
            "iv_error": iv_error,
        }

    except Exception as exc:
        return {
            **diagnostic,
            "implied_vol": np.nan,
            "vega_at_iv": np.nan,
            "iv_status": "ROOT_SOLVER_FAILED",
            "iv_error": str(exc),
        }


# ------------------------------------------------------------
# Baseline inversion demonstration
# ------------------------------------------------------------

baseline_call_iv = implied_volatility(
    option_type=OptionType.CALL,
    price=baseline_call,
    S=BASE.S,
    K=BASE.K,
    tau=BASE.tau,
    r=BASE.r,
    q=BASE.q,
)

baseline_put_iv = implied_volatility(
    option_type=OptionType.PUT,
    price=baseline_put,
    S=BASE.S,
    K=BASE.K,
    tau=BASE.tau,
    r=BASE.r,
    q=BASE.q,
)

iv_baseline_demo = pd.DataFrame([baseline_call_iv, baseline_put_iv])

record_check(
    check="baseline_call_iv_recovery",
    passed=abs(baseline_call_iv["implied_vol"] - BASE.sigma) <= 1e-7,
    detail="Baseline synthetic BSM call price should invert back to the generating volatility.",
    value=baseline_call_iv["implied_vol"],
)

record_check(
    check="baseline_put_iv_recovery",
    passed=abs(baseline_put_iv["implied_vol"] - BASE.sigma) <= 1e-7,
    detail="Baseline synthetic BSM put price should invert back to the generating volatility.",
    value=baseline_put_iv["implied_vol"],
)

iv_baseline_demo[
    [
        "option_type",
        "price",
        "lower_bound",
        "upper_bound",
        "time_value",
        "status",
        "implied_vol",
        "vega_at_iv",
        "iv_status",
        "iv_error",
    ]
]

,option_type,price,lower_bound,upper_bound,time_value,status,implied_vol,vega_at_iv,iv_status,iv_error
0,call,10.45058357,4.87705755,100.00000000,5.57352602,VALID,0.20000000,37.52403469,VALID_IV,None
1,put,5.57352602,0.00000000,95.12294245,5.57352602,VALID,0.20000000,37.52403469,VALID_IV,None


## 4. Synthetic flat-vol recovery test

The baseline call and put both inverted back to the generating volatility:

$$
\sigma_{\mathrm{true}} = 0.20
$$

The next step is to test the inversion engine across many strikes and maturities.

This is a controlled synthetic test:

1. Choose a known volatility.
2. Generate Black-Scholes-Merton prices from that volatility.
3. Invert each generated price back to implied volatility.
4. Check whether the recovered implied volatility matches the original input.

The test uses a grid of strikes and maturities:

- strikes below, near, and above spot,
- short, medium, and longer maturities,
- both calls and puts,
- continuous dividend yield allowed.

The expected result is:

$$
\sigma_{\mathrm{imp}} \approx \sigma_{\mathrm{true}}
$$

for every non-degenerate option.

This is the main unit test for the implied-volatility engine. If synthetic Black-Scholes prices cannot be inverted back to their generating volatility, then the inversion routine is not trustworthy enough for market data.

In [5]:
# ============================================================
# Synthetic flat-vol recovery test
# ============================================================

TRUE_SIGMA_FLAT = 0.20

strike_grid = np.array([80.0, 90.0, 100.0, 110.0, 120.0])
tau_grid = np.array([0.10, 0.25, 0.50, 1.00, 2.00])

recovery_rows: list[dict[str, Any]] = []

for option_type in [OptionType.CALL, OptionType.PUT]:
    for tau in tau_grid:
        for K in strike_grid:
            synthetic_price = float(
                bsm_price(
                    option_type=option_type,
                    S=BASE.S,
                    K=K,
                    tau=tau,
                    r=BASE.r,
                    q=BASE.q,
                    sigma=TRUE_SIGMA_FLAT,
                )
            )

            result = implied_volatility(
                option_type=option_type,
                price=synthetic_price,
                S=BASE.S,
                K=K,
                tau=tau,
                r=BASE.r,
                q=BASE.q,
            )

            recovered_iv = result["implied_vol"]

            if np.isfinite(recovered_iv):
                abs_iv_error = abs(recovered_iv - TRUE_SIGMA_FLAT)
            else:
                abs_iv_error = np.nan

            recovery_rows.append(
                {
                    "option_type": option_type.value,
                    "S": BASE.S,
                    "K": K,
                    "tau": tau,
                    "r": BASE.r,
                    "q": BASE.q,
                    "true_sigma": TRUE_SIGMA_FLAT,
                    "synthetic_price": synthetic_price,
                    "lower_bound": result["lower_bound"],
                    "upper_bound": result["upper_bound"],
                    "time_value": result["time_value"],
                    "recovered_iv": recovered_iv,
                    "abs_iv_error": abs_iv_error,
                    "vega_at_iv": result["vega_at_iv"],
                    "bounds_status": result["status"],
                    "iv_status": result["iv_status"],
                    "iv_error": result["iv_error"],
                }
            )

flat_vol_recovery = pd.DataFrame(recovery_rows)

valid_or_warning = flat_vol_recovery["iv_status"].isin(
    ["VALID_IV", "VALID_LOW_VEGA_WARNING"]
)

max_abs_iv_error = float(
    flat_vol_recovery.loc[valid_or_warning, "abs_iv_error"].max()
)

mean_abs_iv_error = float(
    flat_vol_recovery.loc[valid_or_warning, "abs_iv_error"].mean()
)

num_rows = int(len(flat_vol_recovery))
num_valid = int((flat_vol_recovery["iv_status"] == "VALID_IV").sum())
num_low_vega = int((flat_vol_recovery["iv_status"] == "VALID_LOW_VEGA_WARNING").sum())
num_failed = int((~valid_or_warning).sum())

record_check(
    check="flat_vol_call_put_grid_iv_recovery",
    passed=max_abs_iv_error <= 1e-6,
    detail="Synthetic flat-vol BSM prices should invert back to their generating volatility.",
    value={
        "true_sigma": TRUE_SIGMA_FLAT,
        "max_abs_iv_error": max_abs_iv_error,
        "mean_abs_iv_error": mean_abs_iv_error,
    },
)

record_check(
    check="flat_vol_recovery_no_unexpected_failures",
    passed=num_failed == 0,
    detail="All non-degenerate synthetic flat-vol prices should return a valid implied volatility.",
    value={
        "num_rows": num_rows,
        "num_valid": num_valid,
        "num_low_vega_warnings": num_low_vega,
        "num_failed": num_failed,
    },
)

flat_vol_summary = pd.DataFrame(
    [
        {
            "num_rows": num_rows,
            "num_valid": num_valid,
            "num_low_vega_warnings": num_low_vega,
            "num_failed": num_failed,
            "true_sigma": TRUE_SIGMA_FLAT,
            "max_abs_iv_error": max_abs_iv_error,
            "mean_abs_iv_error": mean_abs_iv_error,
        }
    ]
)

flat_vol_summary

,num_rows,num_valid,num_low_vega_warnings,num_failed,true_sigma,max_abs_iv_error,mean_abs_iv_error
0,50,50,0,0,0.20000000,0.00000000,0.00000000


## 5. Invalid price diagnostics

The flat-vol recovery test confirms that valid synthetic Black-Scholes-Merton prices invert correctly.

The next step is to test invalid prices deliberately.

For each option type, construct prices in five regions:

1. below the lower no-arbitrage bound,
2. exactly at the lower bound,
3. inside the valid range,
4. exactly at the upper bound,
5. above the upper no-arbitrage bound.

The expected behavior is:

| Price location | Expected interpretation |
|---|---|
| Below lower bound | no valid implied volatility |
| At lower bound | zero-volatility degenerate case |
| Inside bounds | valid positive implied volatility |
| At upper bound | infinite-volatility degenerate case |
| Above upper bound | no valid implied volatility |

This section is important because real option-chain data can contain stale quotes, bad mids, crossed markets, or prices that violate model assumptions. The implied-volatility engine must reject those prices before attempting numerical inversion.

In [6]:
# ============================================================
# Invalid price diagnostics
# ============================================================

invalid_price_rows: list[dict[str, Any]] = []

for option_type in [OptionType.CALL, OptionType.PUT]:
    lower, upper = option_bounds(
        option_type=option_type,
        S=BASE.S,
        K=BASE.K,
        tau=BASE.tau,
        r=BASE.r,
        q=BASE.q,
    )

    if option_type == OptionType.CALL:
        valid_inside_price = baseline_call
    else:
        valid_inside_price = baseline_put

    price_cases = [
        {
            "case": "below_lower_bound",
            "price": lower - 0.01,
            "expected_iv_status": f"NO_IV_BELOW_LOWER_BOUND",
        },
        {
            "case": "at_lower_bound",
            "price": lower,
            "expected_iv_status": "ZERO_VOL_DEGENERATE",
        },
        {
            "case": "inside_bounds",
            "price": valid_inside_price,
            "expected_iv_status": "VALID_IV",
        },
        {
            "case": "at_upper_bound",
            "price": upper,
            "expected_iv_status": "INFINITE_VOL_DEGENERATE",
        },
        {
            "case": "above_upper_bound",
            "price": upper + 0.01,
            "expected_iv_status": "NO_IV_ABOVE_UPPER_BOUND",
        },
    ]

    for case_info in price_cases:
        result = implied_volatility(
            option_type=option_type,
            price=case_info["price"],
            S=BASE.S,
            K=BASE.K,
            tau=BASE.tau,
            r=BASE.r,
            q=BASE.q,
        )

        expected_iv_status = case_info["expected_iv_status"]
        observed_iv_status = result["iv_status"]

        invalid_price_rows.append(
            {
                "option_type": option_type.value,
                "case": case_info["case"],
                "price": case_info["price"],
                "lower_bound": result["lower_bound"],
                "upper_bound": result["upper_bound"],
                "time_value": result["time_value"],
                "bounds_status": result["status"],
                "implied_vol": result["implied_vol"],
                "vega_at_iv": result["vega_at_iv"],
                "expected_iv_status": expected_iv_status,
                "observed_iv_status": observed_iv_status,
                "status_match": observed_iv_status == expected_iv_status,
                "iv_error": result["iv_error"],
            }
        )

invalid_price_diagnostics = pd.DataFrame(invalid_price_rows)

all_invalid_cases_classified_correctly = bool(
    invalid_price_diagnostics["status_match"].all()
)

num_invalid_cases = int(len(invalid_price_diagnostics))
num_status_matches = int(invalid_price_diagnostics["status_match"].sum())
num_status_mismatches = int((~invalid_price_diagnostics["status_match"]).sum())

record_check(
    check="invalid_price_cases_classified_correctly",
    passed=all_invalid_cases_classified_correctly,
    detail="Deliberately invalid and degenerate prices should receive the expected implied-volatility status.",
    value={
        "num_cases": num_invalid_cases,
        "num_status_matches": num_status_matches,
        "num_status_mismatches": num_status_mismatches,
    },
)

invalid_price_diagnostics[
    [
        "option_type",
        "case",
        "price",
        "lower_bound",
        "upper_bound",
        "time_value",
        "bounds_status",
        "implied_vol",
        "expected_iv_status",
        "observed_iv_status",
        "status_match",
    ]
]

,option_type,case,price,lower_bound,upper_bound,time_value,bounds_status,implied_vol,expected_iv_status,observed_iv_status,status_match
0,call,below_lower_bound,4.86705755,4.87705755,100.00000000,-0.01000000,BELOW_LOWER_BOUND,NaN,NO_IV_BELOW_LOWER_BOUND,NO_IV_BELOW_LOWER_BOUND,True
1,call,at_lower_bound,4.87705755,4.87705755,100.00000000,0.00000000,AT_INTRINSIC_OR_ZERO_VOL,0.00000000,ZERO_VOL_DEGENERATE,ZERO_VOL_DEGENERATE,True
2,call,inside_bounds,10.45058357,4.87705755,100.00000000,5.57352602,VALID,0.20000000,VALID_IV,VALID_IV,True
3,call,at_upper_bound,100.00000000,4.87705755,100.00000000,95.12294245,NEAR_UPPER_BOUND,inf,INFINITE_VOL_DEGENERATE,INFINITE_VOL_DEGENERATE,True
4,call,above_upper_bound,100.01000000,4.87705755,100.00000000,95.13294245,ABOVE_UPPER_BOUND,NaN,NO_IV_ABOVE_UPPER_BOUND,NO_IV_ABOVE_UPPER_BOUND,True
5,put,below_lower_bound,-0.01000000,0.00000000,95.12294245,-0.01000000,BELOW_LOWER_BOUND,NaN,NO_IV_BELOW_LOWER_BOUND,NO_IV_BELOW_LOWER_BOUND,True
6,put,at_lower_bound,0.00000000,0.00000000,95.12294245,0.00000000,AT_INTRINSIC_OR_ZERO_VOL,0.00000000,ZERO_VOL_DEGENERATE,ZERO_VOL_DEGENERATE,True
7,put,inside_bounds,5.57352602,0.00000000,95.12294245,5.57352602,VALID,0.20000000,VALID_IV,VALID_IV,True
8,put,at_upper_bound,95.12294245,0.00000000,95.12294245,95.12294245,NEAR_UPPER_BOUND,inf,INFINITE_VOL_DEGENERATE,INFINITE_VOL_DEGENERATE,True
9,put,above_upper_bound,95.13294245,0.00000000,95.12294245,95.13294245,ABOVE_UPPER_BOUND,NaN,NO_IV_ABOVE_UPPER_BOUND,NO_IV_ABOVE_UPPER_BOUND,True


## 6. Near-expiry and low-vega failure modes

The invalid-price test shows that the inversion engine rejects prices outside no-arbitrage bounds.

However, passing no-arbitrage bounds is not enough.

Some prices are theoretically valid but numerically fragile because vega is very small. This usually happens when the option has very little remaining time or when the strike is far from spot.

Low-vega cases include:

- very short-dated options,
- deep out-of-the-money calls,
- deep in-the-money calls,
- deep out-of-the-money puts,
- deep in-the-money puts,
- prices very close to discounted intrinsic value,
- prices very close to the upper bound.

The local relationship between price error and implied-volatility error is approximately:

$$
\Delta \sigma \approx \frac{\Delta V}{\mathrm{Vega}}
$$

This means the same small price error can be harmless for an at-the-money option but damaging for a low-vega option.

In this section, the notebook deliberately constructs difficult but valid option prices. The goal is not to make the inversion fail. The goal is to identify where the recovered implied volatility should be treated with caution.

In [7]:
# ============================================================
# Near-expiry and low-vega failure modes
# ============================================================

VERY_LOW_VEGA_THRESHOLD = 1e-3

low_vega_cases = [
    {
        "case": "baseline_atm_call",
        "option_type": OptionType.CALL,
        "K": 100.0,
        "tau": 1.00,
        "sigma": 0.20,
    },
    {
        "case": "very_short_atm_call",
        "option_type": OptionType.CALL,
        "K": 100.0,
        "tau": 1.0 / 365.0,
        "sigma": 0.20,
    },
    {
        "case": "very_short_otm_call",
        "option_type": OptionType.CALL,
        "K": 105.0,
        "tau": 1.0 / 365.0,
        "sigma": 0.20,
    },
    {
        "case": "deep_otm_call",
        "option_type": OptionType.CALL,
        "K": 130.0,
        "tau": 0.25,
        "sigma": 0.20,
    },
    {
        "case": "deep_itm_call",
        "option_type": OptionType.CALL,
        "K": 70.0,
        "tau": 0.25,
        "sigma": 0.20,
    },
    {
        "case": "baseline_atm_put",
        "option_type": OptionType.PUT,
        "K": 100.0,
        "tau": 1.00,
        "sigma": 0.20,
    },
    {
        "case": "very_short_atm_put",
        "option_type": OptionType.PUT,
        "K": 100.0,
        "tau": 1.0 / 365.0,
        "sigma": 0.20,
    },
    {
        "case": "very_short_otm_put",
        "option_type": OptionType.PUT,
        "K": 95.0,
        "tau": 1.0 / 365.0,
        "sigma": 0.20,
    },
    {
        "case": "deep_otm_put",
        "option_type": OptionType.PUT,
        "K": 70.0,
        "tau": 0.25,
        "sigma": 0.20,
    },
    {
        "case": "deep_itm_put",
        "option_type": OptionType.PUT,
        "K": 130.0,
        "tau": 0.25,
        "sigma": 0.20,
    },
]

low_vega_rows: list[dict[str, Any]] = []

for case in low_vega_cases:
    option_type = case["option_type"]
    K = float(case["K"])
    tau = float(case["tau"])
    sigma = float(case["sigma"])

    price = float(
        bsm_price(
            option_type=option_type,
            S=BASE.S,
            K=K,
            tau=tau,
            r=BASE.r,
            q=BASE.q,
            sigma=sigma,
        )
    )

    iv_result = implied_volatility(
        option_type=option_type,
        price=price,
        S=BASE.S,
        K=K,
        tau=tau,
        r=BASE.r,
        q=BASE.q,
        low_vega_tol=VERY_LOW_VEGA_THRESHOLD,
    )

    recovered_iv = iv_result["implied_vol"]
    vega_at_iv = iv_result["vega_at_iv"]

    if np.isfinite(recovered_iv):
        abs_iv_error = abs(recovered_iv - sigma)
    else:
        abs_iv_error = np.nan

    if np.isfinite(vega_at_iv) and vega_at_iv > 0:
        iv_change_per_cent_price_error = 0.01 / vega_at_iv
    else:
        iv_change_per_cent_price_error = np.nan

    moneyness = BASE.S / K
    log_moneyness = log(BASE.S / K)

    low_vega_rows.append(
        {
            "case": case["case"],
            "option_type": option_type.value,
            "S": BASE.S,
            "K": K,
            "moneyness_S_over_K": moneyness,
            "log_moneyness": log_moneyness,
            "tau": tau,
            "true_sigma": sigma,
            "price": price,
            "lower_bound": iv_result["lower_bound"],
            "upper_bound": iv_result["upper_bound"],
            "time_value": iv_result["time_value"],
            "recovered_iv": recovered_iv,
            "abs_iv_error": abs_iv_error,
            "vega_at_iv": vega_at_iv,
            "iv_change_per_0_01_price_error": iv_change_per_cent_price_error,
            "bounds_status": iv_result["status"],
            "iv_status": iv_result["iv_status"],
            "iv_error": iv_result["iv_error"],
            "low_vega_flag": (
                np.isfinite(vega_at_iv)
                and vega_at_iv < VERY_LOW_VEGA_THRESHOLD
            ),
        }
    )

low_vega_diagnostics = pd.DataFrame(low_vega_rows)

recoverable_low_vega = low_vega_diagnostics["iv_status"].isin(
    ["VALID_IV", "VALID_LOW_VEGA_WARNING"]
)

max_low_vega_abs_iv_error = float(
    low_vega_diagnostics.loc[recoverable_low_vega, "abs_iv_error"].max()
)

num_low_vega_cases = int(len(low_vega_diagnostics))
num_recoverable_low_vega_cases = int(recoverable_low_vega.sum())
num_low_vega_warnings = int(
    (low_vega_diagnostics["iv_status"] == "VALID_LOW_VEGA_WARNING").sum()
)
num_low_vega_flags = int(low_vega_diagnostics["low_vega_flag"].sum())

record_check(
    check="near_expiry_low_vega_cases_recover_when_valid",
    passed=max_low_vega_abs_iv_error <= 1e-6,
    detail="Recoverable near-expiry and low-vega synthetic prices should invert back to their generating volatility.",
    value={
        "num_cases": num_low_vega_cases,
        "num_recoverable_cases": num_recoverable_low_vega_cases,
        "max_abs_iv_error": max_low_vega_abs_iv_error,
    },
)

record_check(
    check="low_vega_cases_are_identified",
    passed=num_low_vega_flags > 0,
    detail="At least one deliberately difficult option should be flagged as low-vega.",
    value={
        "num_low_vega_flags": num_low_vega_flags,
        "num_low_vega_warnings": num_low_vega_warnings,
        "threshold": VERY_LOW_VEGA_THRESHOLD,
    },
)

low_vega_diagnostics[
    [
        "case",
        "option_type",
        "K",
        "tau",
        "true_sigma",
        "price",
        "time_value",
        "recovered_iv",
        "abs_iv_error",
        "vega_at_iv",
        "iv_change_per_0_01_price_error",
        "iv_status",
        "low_vega_flag",
    ]
]

,case,option_type,K,tau,true_sigma,price,time_value,recovered_iv,abs_iv_error,vega_at_iv,iv_change_per_0_01_price_error,iv_status,low_vega_flag
0,baseline_atm_call,call,100.00000000,1.00000000,0.20000000,10.45058357,5.57352602,0.20000000,0.00000000,37.52403469,0.00026650,VALID_IV,False
1,very_short_atm_call,call,100.00000000,0.00273973,0.20000000,0.42448596,0.41078826,0.20000000,0.00000000,2.08780895,0.00478971,VALID_IV,False
2,very_short_otm_call,call,105.00000000,0.00273973,0.20000000,0.00000036,0.00000036,0.20000000,0.00000000,0.00004364,229.12607619,VALID_LOW_VEGA_WARNING,True
3,deep_otm_call,call,130.00000000,0.25000000,0.20000000,0.02278029,0.02278029,0.20000000,0.00000000,0.99517077,0.01004853,VALID_IV,False
4,deep_itm_call,call,70.00000000,0.25000000,0.20000000,30.86977675,0.00022279,0.20000000,0.00000000,0.01818315,0.54995973,VALID_IV,False
5,baseline_atm_put,put,100.00000000,1.00000000,0.20000000,5.57352602,5.57352602,0.20000000,0.00000000,37.52403469,0.00026650,VALID_IV,False
6,very_short_atm_put,put,100.00000000,0.00273973,0.20000000,0.41078826,0.41078826,0.20000000,0.00000000,2.08780895,0.00478971,VALID_IV,False
7,very_short_otm_put,put,95.00000000,0.00273973,0.20000000,0.00000009,0.00000009,0.20000000,0.00000000,0.00001168,856.13212327,VALID_LOW_VEGA_WARNING,True
8,deep_otm_put,put,70.00000000,0.25000000,0.20000000,0.00022279,0.00022279,0.20000000,0.00000000,0.01818315,0.54995973,VALID_IV,False
9,deep_itm_put,put,130.00000000,0.25000000,0.20000000,28.40789436,0.02278029,0.20000000,0.00000000,0.99517077,0.01004853,VALID_IV,False


## 7. Price perturbation and implied-volatility sensitivity

The previous section showed that some valid option prices are fragile because vega is small.

This section makes that fragility explicit.

Starting from synthetic Black-Scholes-Merton prices, perturb each price by a small amount and invert the perturbed price back to implied volatility.

The local approximation is:

$$
\Delta \sigma \approx \frac{\Delta V}{\mathrm{Vega}}
$$

where:

- $\Delta V$ is the option price perturbation,
- $\mathrm{Vega}$ is the option's price sensitivity to volatility,
- $\Delta \sigma$ is the approximate implied-volatility change.

This means a one-cent price error can produce very different implied-volatility errors depending on the option region.

For an at-the-money option with high vega, a one-cent price change may barely move implied volatility.

For a short-dated deep out-of-the-money option with very low vega, the same one-cent price change can make implied volatility explode, collapse, or become invalid because the perturbed price crosses a no-arbitrage boundary.

This is why real option-chain cleaning cannot rely only on whether an implied volatility number exists. The inversion must also track time value, vega, moneyness, and price-distance from no-arbitrage bounds.

In [8]:
# ============================================================
# Price perturbation and implied-volatility sensitivity
# ============================================================

PRICE_PERTURBATIONS = np.array([-0.01, -0.001, 0.0, 0.001, 0.01], dtype=float)

perturbation_rows: list[dict[str, Any]] = []

for case in low_vega_cases:
    option_type = case["option_type"]
    K = float(case["K"])
    tau = float(case["tau"])
    true_sigma = float(case["sigma"])

    base_price = float(
        bsm_price(
            option_type=option_type,
            S=BASE.S,
            K=K,
            tau=tau,
            r=BASE.r,
            q=BASE.q,
            sigma=true_sigma,
        )
    )

    lower, upper = option_bounds(
        option_type=option_type,
        S=BASE.S,
        K=K,
        tau=tau,
        r=BASE.r,
        q=BASE.q,
    )

    base_vega = float(
        bsm_vega(
            S=BASE.S,
            K=K,
            tau=tau,
            r=BASE.r,
            q=BASE.q,
            sigma=true_sigma,
        )
    )

    distance_to_lower = base_price - lower
    distance_to_upper = upper - base_price
    nearest_bound_distance = min(distance_to_lower, distance_to_upper)

    for perturbation in PRICE_PERTURBATIONS:
        perturbed_price = base_price + perturbation

        iv_result = implied_volatility(
            option_type=option_type,
            price=perturbed_price,
            S=BASE.S,
            K=K,
            tau=tau,
            r=BASE.r,
            q=BASE.q,
            low_vega_tol=VERY_LOW_VEGA_THRESHOLD,
        )

        perturbed_iv = iv_result["implied_vol"]

        if np.isfinite(perturbed_iv):
            iv_change = perturbed_iv - true_sigma
            abs_iv_change = abs(iv_change)
        else:
            iv_change = np.nan
            abs_iv_change = np.nan

        if base_vega > 0:
            linearized_iv_change = perturbation / base_vega
        else:
            linearized_iv_change = np.nan

        if np.isfinite(iv_change) and np.isfinite(linearized_iv_change):
            linearization_error = iv_change - linearized_iv_change
        else:
            linearization_error = np.nan

        perturbation_rows.append(
            {
                "case": case["case"],
                "option_type": option_type.value,
                "K": K,
                "tau": tau,
                "true_sigma": true_sigma,
                "base_price": base_price,
                "perturbation": perturbation,
                "perturbed_price": perturbed_price,
                "lower_bound": lower,
                "upper_bound": upper,
                "distance_to_lower": distance_to_lower,
                "distance_to_upper": distance_to_upper,
                "nearest_bound_distance": nearest_bound_distance,
                "base_vega": base_vega,
                "linearized_iv_change": linearized_iv_change,
                "perturbed_iv": perturbed_iv,
                "iv_change": iv_change,
                "abs_iv_change": abs_iv_change,
                "linearization_error": linearization_error,
                "bounds_status": iv_result["status"],
                "iv_status": iv_result["iv_status"],
                "iv_error": iv_result["iv_error"],
                "crossed_bounds": iv_result["iv_status"]
                in [
                    "NO_IV_BELOW_LOWER_BOUND",
                    "NO_IV_ABOVE_UPPER_BOUND",
                ],
            }
        )

iv_perturbation_diagnostics = pd.DataFrame(perturbation_rows)

zero_perturbation_rows = iv_perturbation_diagnostics[
    iv_perturbation_diagnostics["perturbation"] == 0.0
]

zero_perturbation_recoverable = zero_perturbation_rows["iv_status"].isin(
    ["VALID_IV", "VALID_LOW_VEGA_WARNING"]
)

zero_perturbation_max_error = float(
    zero_perturbation_rows.loc[zero_perturbation_recoverable, "abs_iv_change"].max()
)

num_perturbation_rows = int(len(iv_perturbation_diagnostics))
num_crossed_bounds = int(iv_perturbation_diagnostics["crossed_bounds"].sum())

valid_perturbed_rows = iv_perturbation_diagnostics[
    iv_perturbation_diagnostics["iv_status"].isin(
        ["VALID_IV", "VALID_LOW_VEGA_WARNING"]
    )
].copy()

largest_valid_iv_moves = (
    valid_perturbed_rows.sort_values("abs_iv_change", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

record_check(
    check="zero_price_perturbation_recovers_true_iv",
    passed=zero_perturbation_max_error <= 1e-6,
    detail="Zero perturbation should recover the original implied volatility for every recoverable case.",
    value={
        "zero_perturbation_max_error": zero_perturbation_max_error,
        "num_zero_perturbation_rows": int(len(zero_perturbation_rows)),
    },
)

record_check(
    check="price_perturbations_can_cross_bounds",
    passed=num_crossed_bounds > 0,
    detail="Some low-time-value options should become invalid after small adverse price perturbations.",
    value={
        "num_perturbation_rows": num_perturbation_rows,
        "num_crossed_bounds": num_crossed_bounds,
    },
)

largest_valid_iv_moves[
    [
        "case",
        "option_type",
        "K",
        "tau",
        "base_price",
        "perturbation",
        "perturbed_price",
        "base_vega",
        "linearized_iv_change",
        "perturbed_iv",
        "iv_change",
        "abs_iv_change",
        "iv_status",
        "crossed_bounds",
    ]
]

,case,option_type,K,tau,base_price,perturbation,perturbed_price,base_vega,linearized_iv_change,perturbed_iv,iv_change,abs_iv_change,iv_status,crossed_bounds
0,very_short_otm_put,put,95.00000000,0.00273973,0.00000009,0.01000000,0.01000009,0.00001168,856.13216327,0.44013262,0.24013262,0.24013262,VALID_IV,False
1,very_short_otm_call,call,105.00000000,0.00273973,0.00000036,0.01000000,0.01000036,0.00004364,229.12607662,0.41667526,0.21667526,0.21667526,VALID_IV,False
2,very_short_otm_put,put,95.00000000,0.00273973,0.00000009,0.00100000,0.00100009,0.00001168,85.61321633,0.34086935,0.14086935,0.14086935,VALID_IV,False
3,very_short_otm_call,call,105.00000000,0.00273973,0.00000036,0.00100000,0.00100036,0.00004364,22.91260766,0.32263331,0.12263331,0.12263331,VALID_IV,False
4,deep_itm_call,call,70.00000000,0.25000000,30.86977675,0.01000000,30.87977675,0.01818315,0.54995973,0.26903064,0.06903064,0.06903064,VALID_IV,False
5,deep_otm_put,put,70.00000000,0.25000000,0.00022279,0.01000000,0.01022279,0.01818315,0.54995973,0.26903064,0.06903064,0.06903064,VALID_IV,False
6,deep_itm_call,call,70.00000000,0.25000000,30.86977675,0.00100000,30.87077675,0.01818315,0.05499597,0.22430287,0.02430287,0.02430287,VALID_IV,False
7,deep_otm_put,put,70.00000000,0.25000000,0.00022279,0.00100000,0.00122279,0.01818315,0.05499597,0.22430287,0.02430287,0.02430287,VALID_IV,False
8,deep_itm_put,put,130.00000000,0.25000000,28.40789436,-0.01000000,28.39789436,0.99517077,-0.01004853,0.18777803,-0.01222197,0.01222197,VALID_IV,False
9,deep_otm_call,call,130.00000000,0.25000000,0.02278029,-0.01000000,0.01278029,0.99517077,-0.01004853,0.18777803,-0.01222197,0.01222197,VALID_IV,False


## 8. Synthetic smile construction

So far, the notebook has used a flat volatility input.

Under flat volatility, every valid synthetic option price should invert back to the same implied volatility, regardless of strike or maturity.

Real markets usually do not behave that way.

Instead, implied volatility often changes with strike and maturity. This produces an implied-volatility surface.

A simple synthetic smile can be constructed by making volatility depend on log-moneyness:

$$
m = \log\left(\frac{K}{F}\right)
$$

where the forward price is:

$$
F = S e^{(r-q)\tau}
$$

A basic smile shape is:

$$
\sigma(m,\tau)
=
\sigma_0
+
a m
+
b m^2
+
c\sqrt{\tau}
$$

The terms have different roles:

| Term | Interpretation |
|---|---|
| $\sigma_0$ | base volatility level |
| $a m$ | skew or directional slope |
| $b m^2$ | smile curvature |
| $c\sqrt{\tau}$ | maturity-dependent level shift |

This is not a calibrated market model. It is a controlled synthetic surface.

The purpose is to verify that the inversion engine can recover a non-flat volatility pattern after prices are generated from strike-dependent and maturity-dependent volatilities.

In [9]:
# ============================================================
# Synthetic smile construction and recovery test
# ============================================================

SMILE_BASE_VOL = 0.16
SMILE_SKEW = -0.08
SMILE_CURVATURE = 0.35
SMILE_TERM_SLOPE = 0.025
SMILE_MIN_VOL = 0.05
SMILE_MAX_VOL = 1.00

smile_strike_grid = np.array([70.0, 80.0, 90.0, 100.0, 110.0, 120.0, 130.0])
smile_tau_grid = np.array([0.25, 0.50, 1.00, 1.50, 2.00])


def forward_price(
    S: float,
    tau: float,
    r: float,
    q: float,
) -> float:
    """
    Forward price under continuous dividend yield.
    """
    return float(S * np.exp((r - q) * tau))


def forward_log_moneyness(
    S: float,
    K: float,
    tau: float,
    r: float,
    q: float,
) -> float:
    """
    Log-moneyness relative to the forward price.
    """
    F = forward_price(S=S, tau=tau, r=r, q=q)
    return float(np.log(K / F))


def synthetic_smile_sigma(
    K: float,
    tau: float,
    S: float,
    r: float,
    q: float,
) -> float:
    """
    Controlled synthetic implied-volatility smile.

    The function is not calibrated to market data. It is only used
    to generate internally consistent option prices for testing.
    """
    m = forward_log_moneyness(
        S=S,
        K=K,
        tau=tau,
        r=r,
        q=q,
    )

    raw_sigma = (
        SMILE_BASE_VOL
        + SMILE_SKEW * m
        + SMILE_CURVATURE * m**2
        + SMILE_TERM_SLOPE * np.sqrt(tau)
    )

    return float(np.clip(raw_sigma, SMILE_MIN_VOL, SMILE_MAX_VOL))


smile_rows: list[dict[str, Any]] = []

for_tau_strike_pairs = [
    (float(tau), float(K))
    for tau in smile_tau_grid
    for K in smile_strike_grid
]

for tau, K in for_tau_strike_pairs:
    true_sigma = synthetic_smile_sigma(
        K=K,
        tau=tau,
        S=BASE.S,
        r=BASE.r,
        q=BASE.q,
    )

    m = forward_log_moneyness(
        S=BASE.S,
        K=K,
        tau=tau,
        r=BASE.r,
        q=BASE.q,
    )

    F = forward_price(
        S=BASE.S,
        tau=tau,
        r=BASE.r,
        q=BASE.q,
    )

    for option_type in [OptionType.CALL, OptionType.PUT]:
        synthetic_price = float(
            bsm_price(
                option_type=option_type,
                S=BASE.S,
                K=K,
                tau=tau,
                r=BASE.r,
                q=BASE.q,
                sigma=true_sigma,
            )
        )

        iv_result = implied_volatility(
            option_type=option_type,
            price=synthetic_price,
            S=BASE.S,
            K=K,
            tau=tau,
            r=BASE.r,
            q=BASE.q,
        )

        recovered_iv = iv_result["implied_vol"]

        if np.isfinite(recovered_iv):
            abs_iv_error = abs(recovered_iv - true_sigma)
        else:
            abs_iv_error = np.nan

        smile_rows.append(
            {
                "option_type": option_type.value,
                "S": BASE.S,
                "K": K,
                "tau": tau,
                "forward": F,
                "forward_log_moneyness": m,
                "true_sigma": true_sigma,
                "synthetic_price": synthetic_price,
                "lower_bound": iv_result["lower_bound"],
                "upper_bound": iv_result["upper_bound"],
                "time_value": iv_result["time_value"],
                "recovered_iv": recovered_iv,
                "abs_iv_error": abs_iv_error,
                "vega_at_iv": iv_result["vega_at_iv"],
                "bounds_status": iv_result["status"],
                "iv_status": iv_result["iv_status"],
                "iv_error": iv_result["iv_error"],
            }
        )

synthetic_smile_surface = pd.DataFrame(smile_rows)

smile_recoverable = synthetic_smile_surface["iv_status"].isin(
    ["VALID_IV", "VALID_LOW_VEGA_WARNING"]
)

max_smile_abs_iv_error = float(
    synthetic_smile_surface.loc[smile_recoverable, "abs_iv_error"].max()
)

mean_smile_abs_iv_error = float(
    synthetic_smile_surface.loc[smile_recoverable, "abs_iv_error"].mean()
)

smile_vol_range = float(
    synthetic_smile_surface["true_sigma"].max()
    - synthetic_smile_surface["true_sigma"].min()
)

smile_parity_pairs = (
    synthetic_smile_surface
    .pivot_table(
        index=["S", "K", "tau", "forward", "forward_log_moneyness", "true_sigma"],
        columns="option_type",
        values="synthetic_price",
        aggfunc="first",
    )
    .reset_index()
)

smile_parity_pairs["parity_gap"] = (
    smile_parity_pairs["call"]
    - smile_parity_pairs["put"]
    - (
        smile_parity_pairs["S"] * np.exp(-BASE.q * smile_parity_pairs["tau"])
        - smile_parity_pairs["K"] * np.exp(-BASE.r * smile_parity_pairs["tau"])
    )
)

max_abs_smile_parity_gap = float(smile_parity_pairs["parity_gap"].abs().max())

num_smile_rows = int(len(synthetic_smile_surface))
num_smile_recoverable = int(smile_recoverable.sum())
num_smile_failed = int((~smile_recoverable).sum())

record_check(
    check="synthetic_smile_iv_recovery",
    passed=max_smile_abs_iv_error <= 1e-6,
    detail="Synthetic smile prices should invert back to their generating strike- and maturity-dependent volatility.",
    value={
        "num_rows": num_smile_rows,
        "num_recoverable": num_smile_recoverable,
        "num_failed": num_smile_failed,
        "max_abs_iv_error": max_smile_abs_iv_error,
        "mean_abs_iv_error": mean_smile_abs_iv_error,
    },
)

record_check(
    check="synthetic_smile_is_nonflat",
    passed=smile_vol_range > 0.01,
    detail="The synthetic smile should create visible variation in implied volatility across strikes and maturities.",
    value={
        "smile_vol_range": smile_vol_range,
        "min_true_sigma": float(synthetic_smile_surface["true_sigma"].min()),
        "max_true_sigma": float(synthetic_smile_surface["true_sigma"].max()),
    },
)

record_check(
    check="synthetic_smile_put_call_parity",
    passed=max_abs_smile_parity_gap <= 1e-10,
    detail="Synthetic call and put prices generated from the same volatility should satisfy put-call parity.",
    value={
        "max_abs_smile_parity_gap": max_abs_smile_parity_gap,
    },
)

synthetic_smile_surface[
    [
        "option_type",
        "K",
        "tau",
        "forward_log_moneyness",
        "true_sigma",
        "synthetic_price",
        "recovered_iv",
        "abs_iv_error",
        "vega_at_iv",
        "iv_status",
    ]
].head(14)

,option_type,K,tau,forward_log_moneyness,true_sigma,synthetic_price,recovered_iv,abs_iv_error,vega_at_iv,iv_status
0,call,70.00000000,0.25000000,-0.36917494,0.24973554,30.87416845,0.24973554,0.00000000,0.20930180,VALID_IV
1,put,70.00000000,0.25000000,-0.36917494,0.24973554,0.00461448,0.24973554,0.00000000,0.20930180,VALID_IV
2,call,80.00000000,0.25000000,-0.23564355,0.21078624,21.03504893,0.21078624,0.00000000,1.45405232,VALID_IV
3,put,80.00000000,0.25000000,-0.23564355,0.21078624,0.04127297,0.21078624,0.00000000,1.45405232,VALID_IV
4,call,90.00000000,0.25000000,-0.11786052,0.18679073,11.55208948,0.18679073,0.00000000,8.47229793,VALID_IV
5,put,90.00000000,0.25000000,-0.11786052,0.18679073,0.43409152,0.18679073,0.00000000,8.47229793,VALID_IV
6,call,100.00000000,0.25000000,-0.01250000,0.17355469,4.09604135,0.17355469,0.00000000,19.59978195,VALID_IV
7,put,100.00000000,0.25000000,-0.01250000,0.17355469,2.85382140,0.17355469,0.00000000,19.59978195,VALID_IV
8,call,110.00000000,0.25000000,0.08281018,0.16827532,0.75242746,0.16827532,0.00000000,12.79760036,VALID_IV
9,put,110.00000000,0.25000000,0.08281018,0.16827532,9.38598551,0.16827532,0.00000000,12.79760036,VALID_IV


In [10]:
# ============================================================
# Synthetic smile surface tables
# ============================================================

smile_call_surface = synthetic_smile_surface[
    synthetic_smile_surface["option_type"] == "call"
].copy()

smile_put_surface = synthetic_smile_surface[
    synthetic_smile_surface["option_type"] == "put"
].copy()

true_smile_table = (
    smile_call_surface
    .pivot_table(
        index="tau",
        columns="K",
        values="true_sigma",
        aggfunc="first",
    )
    .sort_index()
)

recovered_smile_table = (
    smile_call_surface
    .pivot_table(
        index="tau",
        columns="K",
        values="recovered_iv",
        aggfunc="first",
    )
    .sort_index()
)

smile_error_table = (
    smile_call_surface
    .pivot_table(
        index="tau",
        columns="K",
        values="abs_iv_error",
        aggfunc="first",
    )
    .sort_index()
)

smile_price_table = (
    smile_call_surface
    .pivot_table(
        index="tau",
        columns="K",
        values="synthetic_price",
        aggfunc="first",
    )
    .sort_index()
)

smile_vega_table = (
    smile_call_surface
    .pivot_table(
        index="tau",
        columns="K",
        values="vega_at_iv",
        aggfunc="first",
    )
    .sort_index()
)

call_put_iv_match = (
    synthetic_smile_surface
    .pivot_table(
        index=["K", "tau"],
        columns="option_type",
        values="recovered_iv",
        aggfunc="first",
    )
    .reset_index()
)

call_put_iv_match["call_put_iv_gap"] = (
    call_put_iv_match["call"] - call_put_iv_match["put"]
)

max_abs_call_put_iv_gap = float(
    call_put_iv_match["call_put_iv_gap"].abs().max()
)

max_surface_recovery_error = float(smile_error_table.max().max())

smile_shape_summary = pd.DataFrame(
    [
        {
            "surface": "true_sigma",
            "min": float(smile_call_surface["true_sigma"].min()),
            "max": float(smile_call_surface["true_sigma"].max()),
            "range": float(
                smile_call_surface["true_sigma"].max()
                - smile_call_surface["true_sigma"].min()
            ),
            "mean": float(smile_call_surface["true_sigma"].mean()),
        },
        {
            "surface": "recovered_iv",
            "min": float(smile_call_surface["recovered_iv"].min()),
            "max": float(smile_call_surface["recovered_iv"].max()),
            "range": float(
                smile_call_surface["recovered_iv"].max()
                - smile_call_surface["recovered_iv"].min()
            ),
            "mean": float(smile_call_surface["recovered_iv"].mean()),
        },
        {
            "surface": "abs_iv_error",
            "min": float(smile_call_surface["abs_iv_error"].min()),
            "max": float(smile_call_surface["abs_iv_error"].max()),
            "range": float(
                smile_call_surface["abs_iv_error"].max()
                - smile_call_surface["abs_iv_error"].min()
            ),
            "mean": float(smile_call_surface["abs_iv_error"].mean()),
        },
    ]
)

record_check(
    check="synthetic_smile_call_put_iv_match",
    passed=max_abs_call_put_iv_gap <= 1e-10,
    detail="Calls and puts generated from the same synthetic smile should recover the same implied volatility.",
    value={
        "max_abs_call_put_iv_gap": max_abs_call_put_iv_gap,
    },
)

record_check(
    check="synthetic_smile_surface_table_recovery",
    passed=max_surface_recovery_error <= 1e-6,
    detail="The recovered smile table should match the true synthetic smile table.",
    value={
        "max_surface_recovery_error": max_surface_recovery_error,
    },
)

true_smile_table

K,70.00000000,80.00000000,90.00000000,100.00000000,110.00000000,120.00000000,130.00000000
tau,,,,,,,
0.25000000,0.24973554,0.21078624,0.18679073,0.17355469,0.16827532,0.16900805,0.17436211
0.50000000,0.25919818,0.21908048,0.19405436,0.17989642,0.17378309,0.17375447,0.17840816
1.00000000,0.27541857,0.23296407,0.20587675,0.18987500,0.18209374,0.18054242,0.18379536
1.50000000,0.29037276,0.24558146,0.21643293,0.19858737,0.18913818,0.18606417,0.18791636
2.00000000,0.30488254,0.25775444,0.22654471,0.20685534,0.19573822,0.19114151,0.19159295


In [11]:
# ============================================================
# Synthetic smile recovery tables and shape diagnostics
# ============================================================

from IPython.display import display

smile_wing_diagnostics = []

for tau in true_smile_table.index:
    row = true_smile_table.loc[tau]

    left_wing_sigma = float(row.loc[70.0])
    atm_sigma = float(row.loc[100.0])
    right_wing_sigma = float(row.loc[130.0])

    min_strike = float(row.idxmin())
    min_sigma = float(row.min())
    max_strike = float(row.idxmax())
    max_sigma = float(row.max())

    smile_wing_diagnostics.append(
        {
            "tau": float(tau),
            "left_wing_sigma_K70": left_wing_sigma,
            "atm_sigma_K100": atm_sigma,
            "right_wing_sigma_K130": right_wing_sigma,
            "left_minus_atm": left_wing_sigma - atm_sigma,
            "right_minus_atm": right_wing_sigma - atm_sigma,
            "left_minus_right": left_wing_sigma - right_wing_sigma,
            "min_sigma_strike": min_strike,
            "min_sigma": min_sigma,
            "max_sigma_strike": max_strike,
            "max_sigma": max_sigma,
            "cross_sectional_range": max_sigma - min_sigma,
        }
    )

smile_wing_diagnostics = pd.DataFrame(smile_wing_diagnostics)

max_recovered_table_gap = float(
    (recovered_smile_table - true_smile_table).abs().max().max()
)

max_error_table_value = float(smile_error_table.max().max())

all_recovered_iv_positive = bool(
    (synthetic_smile_surface["recovered_iv"] > 0).all()
)

left_wing_above_atm = bool(
    (smile_wing_diagnostics["left_minus_atm"] > 0).all()
)

right_wing_above_atm = bool(
    (smile_wing_diagnostics["right_minus_atm"] > 0).all()
)

left_wing_above_right_wing = bool(
    (smile_wing_diagnostics["left_minus_right"] > 0).all()
)

record_check(
    check="synthetic_smile_recovered_table_matches_true_table",
    passed=max_recovered_table_gap <= 1e-6,
    detail="Recovered implied-volatility surface should match the true synthetic smile surface.",
    value={
        "max_recovered_table_gap": max_recovered_table_gap,
        "max_error_table_value": max_error_table_value,
    },
)

record_check(
    check="synthetic_smile_shape_has_left_skew",
    passed=left_wing_above_atm and left_wing_above_right_wing,
    detail="The chosen synthetic smile parameters should produce a higher left wing than the ATM region and right wing.",
    value={
        "left_wing_above_atm": left_wing_above_atm,
        "right_wing_above_atm": right_wing_above_atm,
        "left_wing_above_right_wing": left_wing_above_right_wing,
    },
)

record_check(
    check="synthetic_smile_recovered_ivs_positive",
    passed=all_recovered_iv_positive,
    detail="All recovered implied volatilities on the synthetic smile surface should be positive.",
    value={
        "all_recovered_iv_positive": all_recovered_iv_positive,
        "min_recovered_iv": float(synthetic_smile_surface["recovered_iv"].min()),
    },
)

print("Recovered implied-volatility table:")
display(recovered_smile_table)

print("Absolute recovery-error table:")
display(smile_error_table)

print("Smile shape diagnostics by maturity:")
smile_wing_diagnostics

Recovered implied-volatility table:


K,70.00000000,80.00000000,90.00000000,100.00000000,110.00000000,120.00000000,130.00000000
tau,,,,,,,
0.25000000,0.24973554,0.21078624,0.18679073,0.17355469,0.16827532,0.16900805,0.17436211
0.50000000,0.25919818,0.21908048,0.19405436,0.17989642,0.17378309,0.17375447,0.17840816
1.00000000,0.27541857,0.23296407,0.20587675,0.18987500,0.18209374,0.18054242,0.18379536
1.50000000,0.29037276,0.24558146,0.21643293,0.19858737,0.18913818,0.18606417,0.18791636
2.00000000,0.30488253,0.25775444,0.22654471,0.20685534,0.19573822,0.19114151,0.19159295


Absolute recovery-error table:


K,70.00000000,80.00000000,90.00000000,100.00000000,110.00000000,120.00000000,130.00000000
tau,,,,,,,
0.25000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
0.50000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
1.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
1.50000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
2.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000


Smile shape diagnostics by maturity:


,tau,left_wing_sigma_K70,atm_sigma_K100,right_wing_sigma_K130,left_minus_atm,right_minus_atm,left_minus_right,min_sigma_strike,min_sigma,max_sigma_strike,max_sigma,cross_sectional_range
0,0.25000000,0.24973554,0.17355469,0.17436211,0.07618086,0.00080742,0.07537343,110.00000000,0.16827532,70.00000000,0.24973554,0.08146022
1,0.50000000,0.25919818,0.17989642,0.17840816,0.07930176,-0.00148826,0.08079003,120.00000000,0.17375447,70.00000000,0.25919818,0.08544371
2,1.00000000,0.27541857,0.18987500,0.18379536,0.08554357,-0.00607964,0.09162321,120.00000000,0.18054242,70.00000000,0.27541857,0.09487615
3,1.50000000,0.29037276,0.19858737,0.18791636,0.09178539,-0.01067101,0.10245640,120.00000000,0.18606417,70.00000000,0.29037276,0.10430859
4,2.00000000,0.30488254,0.20685534,0.19159295,0.09802720,-0.01526239,0.11328958,120.00000000,0.19114151,70.00000000,0.30488254,0.11374103


## 9. Final validation ledger

This notebook built an implied-volatility engine from first principles:

1. European no-arbitrage bounds,
2. Black-Scholes-Merton pricing,
3. vega and monotonicity,
4. implied-volatility inversion,
5. invalid price rejection,
6. low-vega diagnostics,
7. price perturbation sensitivity,
8. synthetic smile recovery.

The core result is that implied volatility should not be treated as a raw number produced by a solver.

It should be treated as a diagnostic object with supporting metadata:

| Field | Purpose |
|---|---|
| price | observed or synthetic option price |
| lower bound | no-arbitrage lower price limit |
| upper bound | no-arbitrage upper price limit |
| time value | distance above discounted intrinsic value |
| implied volatility | volatility that reproduces the price |
| vega | local sensitivity of price to volatility |
| inversion status | whether the solver result is valid, degenerate, or invalid |
| error message | reason for failure, if any |

A valid implied-volatility workflow should answer three questions before using the number:

1. Does the price pass no-arbitrage bounds?
2. Is the implied volatility numerically recoverable?
3. Is the recovered volatility stable enough to trust?

The final ledger below checks whether the notebook passed its internal tests.

In [12]:
# ============================================================
# Final validation ledger
# ============================================================

prior_validation_ledger = pd.DataFrame(validation_rows).copy()

if prior_validation_ledger.empty:
    all_prior_checks_passed = False
    num_prior_checks = 0
    num_prior_passed = 0
    num_prior_failed = 0
else:
    prior_validation_ledger["passed"] = prior_validation_ledger["passed"].astype(bool)

    all_prior_checks_passed = bool(prior_validation_ledger["passed"].all())
    num_prior_checks = int(len(prior_validation_ledger))
    num_prior_passed = int(prior_validation_ledger["passed"].sum())
    num_prior_failed = int((~prior_validation_ledger["passed"]).sum())

record_check(
    check="notebook_06_all_prior_checks_passed",
    passed=all_prior_checks_passed,
    detail="All prior internal validation checks in Notebook 06 should pass.",
    value={
        "num_prior_checks": num_prior_checks,
        "num_prior_passed": num_prior_passed,
        "num_prior_failed": num_prior_failed,
    },
)

validation_ledger = pd.DataFrame(validation_rows).copy()
validation_ledger["passed"] = validation_ledger["passed"].astype(bool)

num_checks = int(len(validation_ledger))
num_passed = int(validation_ledger["passed"].sum())
num_failed = int((~validation_ledger["passed"]).sum())

final_status = "PASS" if num_failed == 0 else "FAIL"

validation_summary = pd.DataFrame(
    [
        {
            "notebook": "06_implied_volatility_inversion_and_no_arbitrage_bounds",
            "num_checks": num_checks,
            "num_passed": num_passed,
            "num_failed": num_failed,
            "final_status": final_status,
        }
    ]
)

failed_validation_checks = validation_ledger[
    ~validation_ledger["passed"]
].copy()

print("Final validation summary:")
display(validation_summary)

print("Validation ledger:")
display(
    validation_ledger[
        [
            "check",
            "passed",
            "detail",
            "value",
        ]
    ]
)

if num_failed > 0:
    print("Failed validation checks:")
    display(
        failed_validation_checks[
            [
                "check",
                "detail",
                "value",
            ]
        ]
    )
else:
    print("All Notebook 06 validation checks passed.")

Final validation summary:


,notebook,num_checks,num_passed,num_failed,final_status
0,06_implied_volatility_inversion_and_no_arbitra...,25,25,0,PASS


Validation ledger:


,check,passed,detail,value
0,call_bounds_order,True,European call lower bound should be nonnegativ...,"{'lower': 4.877057549928594, 'upper': 100.0}"
1,put_bounds_order,True,European put lower bound should be nonnegative...,"{'lower': 0.0, 'upper': 95.1229424500714}"
2,baseline_put_call_parity,True,Synthetic BSM call and put prices should satis...,0.00000000
3,call_price_monotone_in_sigma,True,European call BSM price should be nondecreasin...,None
4,put_price_monotone_in_sigma,True,European put BSM price should be nondecreasing...,None
5,call_prices_inside_bounds,True,BSM call prices across the volatility grid sho...,None
6,put_prices_inside_bounds,True,BSM put prices across the volatility grid shou...,None
7,baseline_call_iv_recovery,True,Baseline synthetic BSM call price should inver...,0.20000000
8,baseline_put_iv_recovery,True,Baseline synthetic BSM put price should invert...,0.20000000
9,flat_vol_call_put_grid_iv_recovery,True,Synthetic flat-vol BSM prices should invert ba...,"{'true_sigma': 0.2, 'max_abs_iv_error': 2.9266..."


All Notebook 06 validation checks passed.
